# Project M

End-to-end notebook for the project:
**What factors predict the need for mechanical ventilation in Parkinson's patients with pneumonia?**

This notebook combines:
- cohort definition and feature extraction from MIMIC-IV,
- dataset building,
- focused EDA,
- validated modeling.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import mannwhitneyu, fisher_exact, chi2_contingency
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import statsmodels.api as sm

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


## 1. Load Required MIMIC-IV Tables

In [ ]:
SHARED_DIR = Path('/content/drive/MyDrive/Третий год/mimic-iv-data')
LOCAL_DIR = Path('/content/mimic-iv')
HOSP = LOCAL_DIR / 'hosp'
ICU = LOCAL_DIR / 'icu'
HOSP_SRC = SHARED_DIR / 'hosp'

MY_FILES = {
    'hosp': [
        'patients.csv.gz',
        'admissions.csv.gz',
        'diagnoses_icd.csv.gz',
        'procedures_icd.csv.gz',
        'emar.csv.gz',
        'labevents.csv.gz',
        'omr.csv.gz',
    ],
    'icu': [
        'icustays.csv.gz',
        'chartevents.csv.gz',
        'procedureevents.csv.gz',
        'inputevents.csv.gz',
    ],
}

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
for module, filenames in MY_FILES.items():
    (LOCAL_DIR / module).mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        src = SHARED_DIR / module / filename
        dst = LOCAL_DIR / module / filename
        if not dst.exists():
            shutil.copy2(src, dst)

print('Files copied to local runtime.')


Files copied to local runtime.


In [ ]:
patients = pd.read_csv(HOSP / 'patients.csv.gz')
admissions = pd.read_csv(HOSP / 'admissions.csv.gz', parse_dates=['admittime', 'dischtime', 'deathtime'])
diagnoses = pd.read_csv(HOSP / 'diagnoses_icd.csv.gz', dtype={'icd_code': str})
procedures_icd = pd.read_csv(HOSP / 'procedures_icd.csv.gz', dtype={'icd_code': str})
icustays = pd.read_csv(ICU / 'icustays.csv.gz', parse_dates=['intime', 'outtime'])
omr = pd.read_csv(HOSP / 'omr.csv.gz')

pd_hadm_ids = set(diagnoses.loc[diagnoses['icd_code'].str.startswith('G20', na=False), 'hadm_id'])
pd_subject_ids = set(diagnoses.loc[diagnoses['icd_code'].str.startswith('G20', na=False), 'subject_id'])
print(f'PD admissions found: {len(pd_hadm_ids):,}')
print(f'PD subjects found: {len(pd_subject_ids):,}')

print('Loading emar...')
emar_chunks = []
for chunk in pd.read_csv(
    HOSP_SRC / 'emar.csv.gz',
    chunksize=500_000,
    usecols=['subject_id', 'hadm_id', 'medication', 'event_txt'],
    dtype={'medication': str},
    low_memory=False,
):
    emar_chunks.append(chunk[chunk['hadm_id'].isin(pd_hadm_ids)])
emar = pd.concat(emar_chunks, ignore_index=True)

LAB_ITEMS = {
    51301: 'wbc',
    50889: 'crp',
    50862: 'albumin',
    50912: 'creatinine',
    50813: 'lactate',
    50983: 'sodium',
    50820: 'ph_arterial',
    50821: 'pao2',
}
print('Loading labevents...')
lab_chunks = []
for chunk in pd.read_csv(
    HOSP / 'labevents.csv.gz',
    chunksize=1_000_000,
    usecols=['subject_id', 'hadm_id', 'itemid', 'valuenum', 'charttime'],
    parse_dates=['charttime'],
):
    mask = chunk['itemid'].isin(LAB_ITEMS) & chunk['hadm_id'].isin(pd_hadm_ids)
    lab_chunks.append(chunk[mask])
labevents = pd.concat(lab_chunks, ignore_index=True)
labevents['lab_name'] = labevents['itemid'].map(LAB_ITEMS)

CHART_ITEMS = {
    220045: 'heart_rate',
    220179: 'sbp',
    223761: 'temp_f',
    223762: 'temp_c',
    220210: 'resp_rate',
    220277: 'spo2',
    223835: 'fio2',
    220739: 'gcs_eye',
    223900: 'gcs_verbal',
    223901: 'gcs_motor',
}
print('Loading chartevents...')
chart_chunks = []
for chunk in pd.read_csv(
    ICU / 'chartevents.csv.gz',
    chunksize=1_000_000,
    usecols=['subject_id', 'hadm_id', 'itemid', 'valuenum', 'charttime'],
    parse_dates=['charttime'],
):
    mask = chunk['itemid'].isin(CHART_ITEMS) & chunk['hadm_id'].isin(pd_hadm_ids)
    chart_chunks.append(chunk[mask])
chartevents = pd.concat(chart_chunks, ignore_index=True)
chartevents['chart_name'] = chartevents['itemid'].map(CHART_ITEMS)

MV_ITEMIDS = {225792, 225794}
print('Loading procedureevents...')
proc_chunks = []
for chunk in pd.read_csv(
    ICU / 'procedureevents.csv.gz',
    chunksize=500_000,
    parse_dates=['starttime', 'endtime'],
):
    mask = chunk['itemid'].isin(MV_ITEMIDS) & chunk['hadm_id'].isin(pd_hadm_ids)
    proc_chunks.append(chunk[mask])
procedureevents = pd.concat(proc_chunks, ignore_index=True)

VASOPRESSOR_ITEMIDS = {221906, 221289, 221662, 222315, 221749}
print('Loading inputevents...')
input_chunks = []
for chunk in pd.read_csv(
    ICU / 'inputevents.csv.gz',
    chunksize=500_000,
    parse_dates=['starttime', 'endtime'],
):
    mask = chunk['itemid'].isin(VASOPRESSOR_ITEMIDS) & chunk['hadm_id'].isin(pd_hadm_ids)
    input_chunks.append(chunk[mask])
inputevents = pd.concat(input_chunks, ignore_index=True)

print('All tables loaded.')


PD admissions found: 2,940
PD subjects found: 1,533
Loading emar...
Loading labevents...


KeyboardInterrupt: 

## 2. Define Cohort and Build Clean Baseline Dataset

In [ ]:
PNEUMONIA_PREFIXES = ['J13', 'J14', 'J15', 'J18', 'J69']
pattern = '|'.join([f'^{p}' for p in PNEUMONIA_PREFIXES])
pneumonia_hadm = set(diagnoses[diagnoses['icd_code'].str.contains(pattern, na=False)]['hadm_id'])

cohort = admissions[admissions['hadm_id'].isin(pd_hadm_ids)].copy()
cohort = cohort[['hadm_id', 'subject_id', 'admittime', 'dischtime', 'hospital_expire_flag']]
cohort = cohort.merge(patients[['subject_id', 'anchor_age', 'anchor_year', 'gender']], on='subject_id', how='left')
cohort['age'] = cohort['anchor_age'] + (cohort['admittime'].dt.year - cohort['anchor_year'])
cohort = cohort[cohort['age'] >= 18].copy()
cohort = cohort[cohort['hadm_id'].isin(pneumonia_hadm)].copy()
hadm_with_data = set(chartevents['hadm_id']) | set(labevents['hadm_id'])
cohort = cohort[cohort['hadm_id'].isin(hadm_with_data)].copy()
cohort = cohort.drop(columns=['anchor_age', 'anchor_year']).reset_index(drop=True)

print(f'Final cohort admissions: {len(cohort):,}')
print(f'Unique patients: {cohort.subject_id.nunique():,}')


In [ ]:
def build_dataset_for_window(baseline_hours=6, min_baseline_before_intubation_hours=2, save_csv=False):
    invasive_mv = procedureevents[
        (procedureevents['itemid'] == 225792) &
        (procedureevents['hadm_id'].isin(set(cohort['hadm_id'])))
    ].copy()

    mv_duration = invasive_mv.copy()
    mv_duration['mv_duration_hours'] = (
        mv_duration['endtime'] - mv_duration['starttime']
    ).dt.total_seconds() / 3600
    first_mv = (
        invasive_mv.sort_values('starttime')
        .groupby('hadm_id', as_index=False)
        .agg(first_intubation_time=('starttime', 'first'))
    )
    mv_duration = mv_duration.groupby('hadm_id', as_index=False)['mv_duration_hours'].sum()
    first_mv = first_mv.merge(mv_duration, on='hadm_id', how='left')
    first_mv['intubation'] = 1

    df = cohort.merge(first_mv, on='hadm_id', how='left')
    df['intubation'] = df['intubation'].fillna(0).astype(int)
    df['mv_duration_hours'] = df['mv_duration_hours'].fillna(0)
    df['hours_to_intubation'] = (
        df['first_intubation_time'] - df['admittime']
    ).dt.total_seconds() / 3600
    df['baseline_end'] = df['admittime'] + pd.Timedelta(hours=baseline_hours)
    mask_int = df['first_intubation_time'].notna()
    df.loc[mask_int, 'baseline_end'] = df.loc[mask_int, ['baseline_end', 'first_intubation_time']].min(axis=1)

    before = len(df)
    df = df[
        df['first_intubation_time'].isna() |
        (df['hours_to_intubation'] >= min_baseline_before_intubation_hours)
    ].copy()
    excluded_early = before - len(df)

    def make_icd_flag(diag_df, prefixes, hadm_ids, col_name):
        pattern = '|'.join([f'^{p}' for p in prefixes])
        matched = set(diag_df[diag_df['icd_code'].str.contains(pattern, na=False)]['hadm_id']) & hadm_ids
        return pd.DataFrame({'hadm_id': list(matched), col_name: 1})

    def make_subject_med_flag(emar_df, keywords, col_name):
        pattern = '|'.join(keywords)
        matched_subjects = set(emar_df[emar_df['medication'].str.contains(pattern, case=False, na=False)]['subject_id'])
        return pd.DataFrame({'subject_id': list(matched_subjects), col_name: 1})

    omr_bmi = omr[omr['result_name'].str.contains('BMI', case=False, na=False)][['subject_id', 'chartdate', 'result_value']].copy()
    omr_bmi['chartdate'] = pd.to_datetime(omr_bmi['chartdate'])
    omr_bmi['result_value'] = pd.to_numeric(omr_bmi['result_value'], errors='coerce')
    omr_bmi = omr_bmi.rename(columns={'result_value': 'bmi'})
    omr_bmi2 = df[['hadm_id', 'subject_id', 'admittime']].merge(omr_bmi, on='subject_id', how='left')
    omr_bmi2 = omr_bmi2[omr_bmi2['chartdate'] <= omr_bmi2['admittime'].dt.normalize()]
    omr_bmi2 = omr_bmi2.sort_values('chartdate').groupby('hadm_id', as_index=False)['bmi'].last()
    df = df.merge(omr_bmi2, on='hadm_id', how='left')

    pd_flag = make_icd_flag(diagnoses, ['G20'], set(df['hadm_id']), 'parkinson_dx')
    df = df.merge(pd_flag, on='hadm_id', how='left')
    df['parkinson_dx'] = df['parkinson_dx'].fillna(0).astype(int)

    for med_df in [
        make_subject_med_flag(emar, ['levodopa', 'carbidopa', 'sinemet'], 'levodopa_use'),
        make_subject_med_flag(emar, ['pramipexole', 'ropinirole', 'rotigotine'], 'dopamine_agonist_use'),
        make_subject_med_flag(emar, ['trihexyphenidyl', 'benztropine'], 'anticholinergic_use'),
    ]:
        df = df.merge(med_df, on='subject_id', how='left')
    for col in ['levodopa_use', 'dopamine_agonist_use', 'anticholinergic_use']:
        df[col] = df[col].fillna(0).astype(int)

    if 'subject_id' in procedures_icd.columns:
        dbs_subjects = set(procedures_icd.loc[procedures_icd['icd_code'].isin(['0293', '0NH60MZ']), 'subject_id'])
        df['dbs_history'] = df['subject_id'].isin(dbs_subjects).astype(int)
    else:
        dbs_hadm = set(procedures_icd.loc[procedures_icd['icd_code'].isin(['0293', '0NH60MZ']), 'hadm_id'])
        df['dbs_history'] = df['hadm_id'].isin(dbs_hadm).astype(int)

    for prefixes, col_name in [
        (['J69'], 'aspiration_pneumonia'),
        (['J13', 'J14', 'J15', 'J18', 'J69'], 'pneumonia_any'),
        (['R13', 'J69'], 'dysphagia')
    ]:
        flag_df = make_icd_flag(diagnoses, prefixes, set(df['hadm_id']), col_name)
        df = df.merge(flag_df, on='hadm_id', how='left')
        df[col_name] = df[col_name].fillna(0).astype(int)

    adm_lookup = admissions[['hadm_id', 'subject_id', 'admittime']].copy()
    diag_with_time = diagnoses.merge(adm_lookup, on=['hadm_id', 'subject_id'], how='left')
    current_admissions = df[['hadm_id', 'subject_id', 'admittime']].rename(columns={'hadm_id': 'index_hadm_id', 'admittime': 'index_admittime'})
    prior_diag = current_admissions.merge(diag_with_time, on='subject_id', how='left')
    prior_diag = prior_diag[prior_diag['admittime'] <= prior_diag['index_admittime']].copy()

    for prefixes, col_name in [
        (['F00', 'F01', 'F02', 'F03', 'G30', 'G31'], 'dementia'),
        (['E10', 'E11', 'E12', 'E13'], 'diabetes'),
        (['I50'], 'heart_failure'),
        (['J44', 'J45'], 'copd_asthma'),
        (['E41', 'E42', 'E43', 'E44', 'E45', 'E46'], 'malnutrition'),
        (['N390'], 'uti'),
        (['I10', 'I11', 'I12', 'I13'], 'hypertension'),
        (['N18', 'N19'], 'renal_failure'),
        (['F17', 'Z72.0', 'Z87.891'], 'smoking_icd')
    ]:
        mask = prior_diag['icd_code'].fillna('').astype(str).apply(lambda x: any(x.startswith(p) for p in prefixes))
        flagged = prior_diag.loc[mask, ['index_hadm_id']].drop_duplicates().rename(columns={'index_hadm_id': 'hadm_id'})
        flagged[col_name] = 1
        df = df.merge(flagged, on='hadm_id', how='left')
        df[col_name] = df[col_name].fillna(0).astype(int)

    charlson_map = {
        'mi': (['I21', 'I22', 'I25.2'], 1),
        'chf': (['I50'], 1),
        'pvd': (['I70', 'I71', 'I73', 'I77'], 1),
        'stroke': (['G45', 'G46', 'I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67', 'I68', 'I69'], 1),
        'dementia_cci': (['F00', 'F01', 'F02', 'F03', 'G30'], 1),
        'copd_cci': (['J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47'], 1),
        'rheum': (['M05', 'M06', 'M32', 'M33', 'M34', 'M35', 'M36'], 1),
        'pud': (['K25', 'K26', 'K27', 'K28'], 1),
        'liver_mild': (['B18', 'K70', 'K71', 'K73', 'K74', 'K76'], 1),
        'diabetes_cci': (['E10', 'E11', 'E12', 'E13', 'E14'], 1),
        'hemiplegia': (['G81', 'G82', 'G83'], 2),
        'renal': (['N18', 'N19', 'I12', 'I13'], 2),
        'diabetes_comp': (['E10.2', 'E10.3', 'E10.4', 'E10.5', 'E11.2', 'E11.3', 'E11.4', 'E11.5'], 2),
        'cancer': (['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C70', 'C71', 'C72', 'C73', 'C74', 'C75'], 2),
        'liver_severe': (['K70.4', 'K71.1', 'K72', 'K76.5', 'K76.6', 'K76.7'], 3),
        'metastatic': (['C77', 'C78', 'C79', 'C80'], 6),
        'aids': (['B20', 'B21', 'B22', 'B24'], 6),
    }
    cci_rows = []
    for hadm_id, group in prior_diag.groupby('index_hadm_id'):
        codes = set(group['icd_code'].dropna().astype(str).tolist())
        score = 0
        for _, (prefixes, weight) in charlson_map.items():
            if any(any(code.startswith(p) for p in prefixes) for code in codes):
                score += weight
        cci_rows.append({'hadm_id': hadm_id, 'charlson_index': score})
    cci_df = pd.DataFrame(cci_rows)
    df = df.merge(cci_df, on='hadm_id', how='left')
    df['charlson_index'] = df['charlson_index'].fillna(0).astype(int)

    chart_adm = chartevents.merge(df[['hadm_id', 'admittime', 'baseline_end']], on='hadm_id', how='inner')
    chart_adm = chart_adm[
        (chart_adm['charttime'] >= chart_adm['admittime']) &
        (chart_adm['charttime'] <= chart_adm['baseline_end'])
    ].copy()
    vitals_mean = (
        chart_adm[chart_adm['chart_name'].isin(['heart_rate', 'sbp', 'resp_rate', 'spo2', 'temp_f', 'temp_c'])]
        .groupby(['hadm_id', 'chart_name'])['valuenum']
        .mean()
        .unstack('chart_name')
        .reset_index()
    )
    vitals_first = (
        chart_adm[chart_adm['chart_name'].isin(['fio2', 'gcs_eye', 'gcs_verbal', 'gcs_motor'])]
        .sort_values('charttime')
        .groupby(['hadm_id', 'chart_name'])['valuenum']
        .first()
        .unstack('chart_name')
        .reset_index()
    )
    vitals = vitals_mean.merge(vitals_first, on='hadm_id', how='outer')
    for col in ['temp_f', 'temp_c', 'gcs_eye', 'gcs_verbal', 'gcs_motor']:
        if col not in vitals.columns:
            vitals[col] = np.nan
    vitals['temperature_c'] = vitals['temp_c'].fillna((vitals['temp_f'] - 32) * 5 / 9)
    vitals['gcs_total'] = vitals[['gcs_eye', 'gcs_verbal', 'gcs_motor']].sum(axis=1, min_count=3)
    vitals = vitals.drop(columns=['temp_f', 'temp_c', 'gcs_eye', 'gcs_verbal', 'gcs_motor'])
    df = df.merge(vitals, on='hadm_id', how='left')

    lab_adm = labevents.merge(df[['hadm_id', 'admittime', 'baseline_end']], on='hadm_id', how='inner')
    lab_adm = lab_adm[
        (lab_adm['charttime'] >= lab_adm['admittime']) &
        (lab_adm['charttime'] <= lab_adm['baseline_end'])
    ].copy()
    labs = (
        lab_adm.sort_values('charttime')
        .groupby(['hadm_id', 'lab_name'])['valuenum']
        .first()
        .unstack('lab_name')
        .reset_index()
    )
    df = df.merge(labs, on='hadm_id', how='left')
    for col in ['wbc', 'crp', 'albumin', 'creatinine', 'lactate', 'sodium', 'ph_arterial', 'pao2']:
        if col not in df.columns:
            df[col] = np.nan
    df['fio2_fraction'] = df['fio2'].apply(lambda x: x / 100 if pd.notna(x) and x > 1 else x)
    df['pf_ratio'] = np.where(df['pao2'].notna() & df['fio2_fraction'].notna(), df['pao2'] / df['fio2_fraction'], np.nan)
    df = df.drop(columns=['fio2_fraction'])

    icu_first = (
        icustays[icustays['hadm_id'].isin(set(df['hadm_id']))]
        .groupby('hadm_id', as_index=False)['intime']
        .min()
        .rename(columns={'intime': 'first_icu_intime'})
    )
    df = df.merge(icu_first, on='hadm_id', how='left')
    df['icu_admission_baseline'] = (
        df['first_icu_intime'].notna() & (df['first_icu_intime'] <= df['baseline_end'])
    ).astype(int)

    vaso = inputevents[inputevents['hadm_id'].isin(set(df['hadm_id']))].copy()
    if not vaso.empty:
        vaso_first = vaso.groupby('hadm_id', as_index=False)['starttime'].min().rename(columns={'starttime': 'first_vaso_time'})
        df = df.merge(vaso_first, on='hadm_id', how='left')
        df['vasopressor_baseline'] = (
            df['first_vaso_time'].notna() & (df['first_vaso_time'] <= df['baseline_end'])
        ).astype(int)
    else:
        df['vasopressor_baseline'] = 0

    col_order = [
        'hadm_id', 'subject_id',
        'age', 'gender', 'bmi',
        'parkinson_dx',
        'levodopa_use', 'dopamine_agonist_use', 'anticholinergic_use', 'dbs_history',
        'intubation', 'first_intubation_time', 'hours_to_intubation', 'mv_duration_hours',
        'aspiration_pneumonia', 'pneumonia_any', 'dysphagia',
        'heart_rate', 'sbp', 'temperature_c', 'resp_rate', 'spo2', 'fio2', 'gcs_total',
        'wbc', 'crp', 'albumin', 'creatinine', 'lactate', 'sodium', 'ph_arterial', 'pao2', 'pf_ratio',
        'dementia', 'diabetes', 'heart_failure', 'copd_asthma', 'malnutrition', 'uti',
        'hypertension', 'renal_failure', 'smoking_icd', 'charlson_index',
        'icu_admission_baseline', 'vasopressor_baseline',
        'admittime', 'baseline_end', 'dischtime', 'hospital_expire_flag'
    ]
    col_order = [c for c in col_order if c in df.columns]
    df = df[col_order].reset_index(drop=True)

    metadata = {
        'baseline_hours': baseline_hours,
        'excluded_early_intubations': excluded_early,
        'rows': len(df),
        'events': int(df['intubation'].sum()),
    }

    if save_csv:
        output_path = f'/content/my_dataset_{baseline_hours}h.csv'
        df.to_csv(output_path, index=False)
        print(f'Saved extracted dataset to {output_path}')

    return df, metadata


BASELINE_HOURS = 6
MIN_BASELINE_BEFORE_INTUBATION_HOURS = 2

saved_datasets = {}
for hrs in [6, 12, 24]:
    df_tmp, meta_tmp = build_dataset_for_window(hrs, MIN_BASELINE_BEFORE_INTUBATION_HOURS, save_csv=True)
    saved_datasets[hrs] = meta_tmp

df, extraction_meta = build_dataset_for_window(BASELINE_HOURS, MIN_BASELINE_BEFORE_INTUBATION_HOURS, save_csv=False)
print('Saved dataset summaries by window:')
print(saved_datasets)
print('Main analysis extraction summary:', extraction_meta)
display(df.head())
